# Exercise 3: Double Integrator Motion Simulation

The goal of this exercise is to build a Python class that defines the dynamics of a double integrator.

Unlike in the `nonholonomic_wheeled_robot_dynamics.ipynb` notebook, here we will leverage *batching* to rollout several trajectories simultaneously.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

#### Double Integrator Dynamics

The double integrator dynamics are:
\begin{equation}
\dot{x} = v_x, \quad \dot{y} = v_y, \quad \dot{v}_x = a_x, \quad \dot{v}_y = a_y.
\end{equation}
where $(x,y)$ is the position, $(v_x, v_y)$ is the velocity, and $(a_x, a_y)$ is the acceleration control input.

### Exercise 3.1 and 3.2: Define Double Integrator Dynamics Class Using Batching

Complete the implementations of the `step` and `rollout` functions. Note that in the `step` method you should leverage the variables `A` and `B` to define a single equation with batching to compute multiple trajectories in parallel.

In [ ]:
from dynamics import Dynamics
class DoubleIntegratorDynamics(Dynamics):
    def __init__(self) -> None:
        super().__init__()
        self.xdd_max = 0.5 # m/s^2
        self.ydd_max = 0.5 # m/s^2
        self.n = 4
        self.m = 2

    def step(self, x_batch: np.ndarray, u: np.ndarray) -> np.ndarray:
        """
        Compute the next state from the current state and control.

        Args:
            x_batch: batch of current states, [x, y, vx, vy], of each rollout, shape (n, num_rollouts)
            u_batch: control input, [ax, ay], shape (m,)

        Returns:
            Batch of next states, `self.dt` seconds in the future, shape (n, num_rollouts)
        """
        num_rollouts = x_batch.shape[1]
    
        # Define Gaussian Noise
        if self.noisy == False:
          var = np.array([0.0, 0.0, 0.0, 0.0])
        elif self.noisy == True:
          var = np.array([0.01, 0.01, 0.001, 0.001])

        # Batch the noise
        var_batch = np.tile(var[:, None], (1, num_rollouts))
        w_batch = np.random.normal(loc=np.zeros((self.n, num_rollouts)), scale=var_batch)
    
        # State space dynamics
        A = np.array([[1.0, 0.0, self.dt, 0.0],
                      [0.0, 1.0, 0.0, self.dt],
                      [0.0, 0.0, 1.0, 0.0],
                      [0.0, 0.0, 0.0, 1.0]])
    
        B = np.array([[0.0, 0.0],
                      [0.0, 0.0],
                      [self.dt, 0.0],
                      [0.0, self.dt]])
    
        ##### YOUR CODE STARTS HERE #####
        # Construct `x_batch_next` by applying discrete time dynamics vectorized equations. 
        # Will require use of `A` and `B`
        raise NotImplementedError("Need to implement code here.")
        ###### YOUR CODE ENDS HERE ######
    
        # Add noise
        x_batch_next += w_batch
    
        return x_batch_next

    def rollout(self, x0: np.ndarray, u_sequence: np.ndarray, num_rollouts: int) -> np.ndarray:
        """
        Rollout the trajectory from x0 using the given control sequence.

        Args:
            x0: initial state, [x, y, vx, vy]
            u_sequence: sequence of control inputs to apply, shape (m, num_steps)
            num_rollouts: number of rollouts to perform

        Returns:
            Array of rollouts, shape (num_steps, n, num_rollouts)
        """
        num_steps = u_sequence.shape[1]
        
        rollouts = np.zeros((num_steps + 1, self.n, num_rollouts))
        rollouts[0, :, :] = np.tile(x0[:, None], (1, num_rollouts))
        ##### YOUR CODE STARTS HERE #####
        # Use ONLY one for-loop to loop `num_steps` to populate `rollouts`. 
        # Use the `step` function above.
        raise NotImplementedError("Need to implement code here.")
        ###### YOUR CODE ENDS HERE ######
    
        return rollouts

Run the script below to simulate the double integrator motion and plot the state and control trajectories.

In [ ]:
# Constants
num_rollouts = 10
num_steps = 100

# Define the dynamics class
dynamics = DoubleIntegratorDynamics()

# Define the control inputs
u_sequence = np.zeros((dynamics.m, num_steps))
u_sequence[0,:] = np.sin(np.linspace(0, 2 * np.pi, num_steps))
u_sequence[1,:] = np.cos(np.linspace(0, 2 * np.pi, num_steps))

# Define the initial state
x0 = np.array([0, 0, 0, 0])

# Rollouts dynamics
rollouts = dynamics.rollout(x0, u_sequence, num_rollouts)

# Plot control inputs
fig_ctrl, axs_ctrl = plt.subplots(nrows=dynamics.m)
ylabels_ctrl = ["xdd", "ydd"]
ylims_ctrl = np.array([[-1.1, 1.1], [-1.1, 1.1]])
for j in range(dynamics.m):
  axs_ctrl[j].plot(np.linspace(0, num_steps, num_steps), u_sequence[j, :])
  axs_ctrl[j].set_ylabel(ylabels_ctrl[j])
  axs_ctrl[j].set_ylim(ylims_ctrl[j, :])
fig_ctrl.suptitle("Control")

# Plots state trajectory rollouts
fig_state, axs_state = plt.subplots(nrows=dynamics.n)
ylabels_state = ["x", "y", "xd", "yd"]
ylims_state = np.array([[-0.5, 0.5], [-0.5, 0.5], [-0.1, 0.5], [-0.3, 0.3]])
for i in range(num_rollouts):
  for j in range(dynamics.n):
    axs_state[j].plot(np.linspace(0, num_steps + 1, num_steps + 1), rollouts[:, j, i], c='c')
    axs_state[j].set_ylabel(ylabels_state[j])
    axs_state[j].set_ylim(ylims_state[j, :])
fig_state.suptitle("State")
plt.show()